# Experimento 2

## Importar librerías y cargar datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Cargar dataset preparado de fase de limpieza
df = pd.read_parquet("nhanes_limpio.parquet")
print(f"Dataset original: {df.shape}")

# Variable objetivo
le_target = LabelEncoder()
y = le_target.fit_transform(df["Diabetes"])
X = df.drop(columns=["Diabetes"])

# División estratificada en train/test
X_train, X_test, y_train_arr, y_test_arr = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convertir y_train e y_test a Series con los mismos índices que X_train y X_test
y_train = pd.Series(y_train_arr, index=X_train.index)
y_test = pd.Series(y_test_arr, index=X_test.index)

# Separar variables categóricas y numéricas
categorical_vars = X.select_dtypes(include='category').columns.tolist()
numerical_vars = X.select_dtypes(include=["float64", "int32"]).columns.tolist()

print(f"Variables categóricas ({len(categorical_vars)}): {categorical_vars}")
print(f"Variables numéricas ({len(numerical_vars)}): {numerical_vars}")

## RandomForestClassifer para ver las variables más importantes

In [ ]:
# Paso 1: Preprocesamiento (OneHot para categóricas, escalar numéricas)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_vars),
        ("num", StandardScaler(), numerical_vars),
    ]
)

# Paso 2: Pipeline completo con RandomForest
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Paso 3: Entrenamiento
pipeline.fit(X_train, y_train)

# Paso 4: Obtener nombres de features tras preprocesamiento
ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_vars)
all_feature_names = list(cat_feature_names) + numerical_vars

# Paso 5: Obtener importancias del modelo
importances = pipeline.named_steps["clf"].feature_importances_

# Paso 6: Agrupar importancias por variable original
importance_dict = {}

# Variables categóricas: agrupar por nombre base
for var in categorical_vars:
    # Todas las columnas que empiezan por el nombre de la variable y "_"
    indices = [i for i, name in enumerate(all_feature_names) if name.startswith(f"{var}_")]
    total_importance = sum(importances[i] for i in indices)
    importance_dict[var] = total_importance

# Variables numéricas: van directamente
for var in numerical_vars:
    idx = all_feature_names.index(var)
    importance_dict[var] = importances[idx]

# Paso 7: Convertir a DataFrame ordenado
importances_grouped = pd.DataFrame({
    "Feature": list(importance_dict.keys()),
    "Importance": list(importance_dict.values())
}).sort_values(by="Importance", ascending=False).head(20)

# Paso 8: Gráfico
plt.figure(figsize=(10, 8))
sns.barplot(x="Importance", y="Feature", data=importances_grouped)
plt.title("Top 20 Variables más Importantes (agrupadas)")
plt.tight_layout()
plt.show()


In [ ]:
# Paso 1: Obtener nombres de todas las variables tras el preprocesamiento
ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_vars)
all_feature_names = list(cat_feature_names) + numerical_vars

# Paso 2: Obtener importancias del modelo
importances = pipeline.named_steps["clf"].feature_importances_

# Paso 3: Agrupar importancias por variable original
importance_dict = {}

for var in categorical_vars:
    indices = [i for i, name in enumerate(all_feature_names) if name.startswith(f"{var}_")]
    total_importance = sum(importances[i] for i in indices)
    importance_dict[var] = total_importance

for var in numerical_vars:
    idx = all_feature_names.index(var)
    importance_dict[var] = importances[idx]

# Paso 4: Convertir a DataFrame
importances_grouped = pd.DataFrame({
    "Feature": list(importance_dict.keys()),
    "Importance": list(importance_dict.values())
})

# Paso 5: Filtrar por importancia > 0.03
importantes_003 = importances_grouped[importances_grouped["Importance"] > 0.03]
importantes_003_sorted = importantes_003.sort_values(by="Importance", ascending=False)

# Paso 6: Visualizar
plt.figure(figsize=(10, 6))
sns.barplot(x="Importance", y="Feature", data=importantes_003_sorted)
plt.title("Variables con Importancia > 0.03")
plt.tight_layout()
plt.show()

# Paso 7: Crear listas de variables para simular NA
vars_003 = importantes_003_sorted["Feature"].tolist()
num_sim_na = [var for var in vars_003 if var in numerical_vars]
cat_sim_na = [var for var in vars_003 if var in categorical_vars]

# Mostrar listas con conteo incluido
print(f"Variables numéricas seleccionadas (num_sim_na) [{len(num_sim_na)}]: {num_sim_na}")
print(f"Variables categóricas seleccionadas (cat_sim_na) [{len(cat_sim_na)}]: {cat_sim_na}")



### Conclusión selección variables a imputar faltantes

**Nos quedamos con 15 variables, 12 numéricas y 3 categóricas para simular faltantes.**

Se almacenan en dos listas: num_sim_na y cat_sim_na.

## Ver faltantes en dataset actual, antes de añadir simulados

In [ ]:
# Verificar dimensiones y porcentaje de NA original
n_filas, n_columnas = df.shape
porcentaje_na = df.isna().mean() * 100
na_summary = porcentaje_na[porcentaje_na > 0].sort_values(ascending=False)

# Mostrar resumen de NAs existentes
na_summary_df = pd.DataFrame({
    "Variable": na_summary.index,
    "Porcentaje_NA": na_summary.values
})

print("Resumen de variables con valores faltantes (porcentaje):")
display(na_summary_df)

# Mostrar dimensiones del dataset
print(f"\nDimensiones del dataset: {n_filas} filas, {n_columnas} columnas")

# Obtener variables con NA (ya están en na_summary.index)
con_na = na_summary.index.tolist()

# Obtener variables sin NA (excluyendo la variable respuesta)
sin_na = [col for col in df.columns if col not in con_na and col != "Diabetes"]

# Variable objetivo
variable_respuesta = "Diabetes"

# Verificación final
total_variables = len(con_na) + len(sin_na) + 1  # +1 por la variable respuesta
print(f"\nTotales: {len(con_na)} con NA, {len(sin_na)} sin NA, 1 respuesta → Total = {total_variables} variables")

# Mostrar listas
print(f"\nVariables con NA ({len(con_na)}): {con_na}")
print(f"Variables sin NA ({len(sin_na)}): {sin_na}")
print(f"Variable respuesta: {variable_respuesta}")




## Simular MCAR

In [ ]:
def introducir_mcar(df_original, variables, porcentaje, semilla=42, verbose=True):
    np.random.seed(semilla)
    df_mcar = df_original.copy()
    n_total = df_mcar.shape[0]

    for var in variables:
        n_missing = int(n_total * porcentaje)

        # Evitar sobreescribir NAs existentes
        idx_validos = df_mcar[df_mcar[var].notna()].index
        idx_missing = np.random.choice(idx_validos, size=min(n_missing, len(idx_validos)), replace=False)

        df_mcar.loc[idx_missing, var] = np.nan

        if verbose:
            print(f"Introducidos NA en '{var}': {len(idx_missing)} filas ({len(idx_missing)/n_total:.2%})")

    return df_mcar


In [ ]:
# Unir variables numéricas y categóricas seleccionadas
variables_mcar = num_sim_na + cat_sim_na

# Aplicar faltantes MCAR al 10%
df_mcar_10 = introducir_mcar(df, variables_mcar, porcentaje=0.10, semilla=42, verbose=False)

# Aplicar faltantes MCAR al 20%
df_mcar_20 = introducir_mcar(df, variables_mcar, porcentaje=0.20, semilla=123, verbose=False)

# Aplicar faltantes MCAR al 30%
df_mcar_30 = introducir_mcar(df, variables_mcar, porcentaje=0.30, semilla=999, verbose=False)

# Mostrar resumen de valores faltantes
print("\nValores faltantes por variable (10% MCAR):")
for var in variables_mcar:
    na_count = df_mcar_10[var].isnull().sum()
    na_percent = na_count / len(df_mcar_10) * 100
    print(f"{var:20}: {na_count} / {len(df_mcar_10)} → {na_percent:.2f}%")

# Verificar dimensiones y porcentaje de NA total
n_filas_mcar_10, n_columnas_mcar_10 = df_mcar_10.shape
porcentaje_na_mcar_10 = df_mcar_10.isna().mean() * 100
na_summary_mcar_10 = porcentaje_na_mcar_10[porcentaje_na_mcar_10 > 0].sort_values(ascending=False)

# Mostrar resumen de NAs existentes
na_summary_mcar_10_df = pd.DataFrame({
    "Variable": na_summary_mcar_10.index,
    "Porcentaje_NA": na_summary_mcar_10.values.round(2)
})

print("\nResumen de las 15 variables con más valores faltantes (%) (10% MCAR):")
print(na_summary_mcar_10_df.head(15))

print("\nValores faltantes por variable (20% MCAR):")
for var in variables_mcar:
    na_count = df_mcar_20[var].isnull().sum()
    na_percent = na_count / len(df_mcar_10) * 100
    print(f"{var:20}: {na_count} / {len(df_mcar_10)} → {na_percent:.2f}%")

# Verificar dimensiones y porcentaje de NA total
n_filas_mcar_20, n_columnas_mcar_20 = df_mcar_20.shape
porcentaje_na_mcar_20 = df_mcar_20.isna().mean() * 100
na_summary_mcar_20 = porcentaje_na_mcar_20[porcentaje_na_mcar_20 > 0].sort_values(ascending=False)

# Mostrar resumen de NAs existentes
na_summary_mcar_20_df = pd.DataFrame({
    "Variable": na_summary_mcar_20.index,
    "Porcentaje_NA": na_summary_mcar_20.values.round(2)
})

print("\nResumen de las 15 variables con más valores faltantes (%) (20% MCAR):")
print(na_summary_mcar_20_df.head(15))

print("\nValores faltantes por variable (30% MCAR):")
for var in variables_mcar:
    na_count = df_mcar_30[var].isnull().sum()
    na_percent = na_count / len(df_mcar_10) * 100
    print(f"{var:20}: {na_count} / {len(df_mcar_10)} → {na_percent:.2f}%")

# Verificar dimensiones y porcentaje de NA total
n_filas_mcar_30, n_columnas_mcar_30 = df_mcar_30.shape
porcentaje_na_mcar_30 = df_mcar_30.isna().mean() * 100
na_summary_mcar_30 = porcentaje_na_mcar_30[porcentaje_na_mcar_30 > 0].sort_values(ascending=False)

# Mostrar resumen de NAs existentes
na_summary_mcar_30_df = pd.DataFrame({
    "Variable": na_summary_mcar_30.index,
    "Porcentaje_NA": na_summary_mcar_30.values.round(2)
})

print("\nResumen de las 15 variables con más valores faltantes (%) (30% MCAR):")
print(na_summary_mcar_30_df.head(15))



## Simular MAR

In [ ]:
# Renombrar las listas originales para MCAR (para mantener trazabilidad)
mcar_num_sim_na = num_sim_na
mcar_cat_sim_na = cat_sim_na


In [ ]:
# Variables que se usarán como condicionantes MAR
condicionantes_mar = ["Age", "Poverty", "HealthGen", "Education"]

# Crear las nuevas listas excluyendo las variables condicionantes
mar_num_sim_na = [var for var in mcar_num_sim_na if var not in condicionantes_mar]
mar_cat_sim_na = [var for var in mcar_cat_sim_na if var not in condicionantes_mar]

In [ ]:
print(f"Variables numéricas para simular MAR ({len(mar_num_sim_na)}): {mar_num_sim_na}")
print(f"Variables categóricas para simular MAR ({len(mar_cat_sim_na)}): {mar_cat_sim_na}")


In [ ]:
# Condición 1: Edad mayor o igual a 60
cond_age = (df["Age"] >= 60) & df["Age"].notna()

# Condición 2: Pobreza en el cuartil inferior (25% más pobres)
cond_poverty = (df["Poverty"] <= df["Poverty"].quantile(0.25)) & df["Poverty"].notna()

# Condición 3: Salud autopercibida = "Poor"
cond_healthgen = df["HealthGen"].isin(["Poor"]) & df["HealthGen"].notna()

# Condición 4: Educación baja (8th Grade)
cond_education = df["Education"].isin(["8th Grade"]) & df["Education"].notna()

# Total que cumple al menos una condición (ignora las que tienen NA en todas)
cond_total = cond_age | cond_poverty | cond_healthgen | cond_education

# Cálculos individuales
n_age = cond_age.sum()
n_poverty = cond_poverty.sum()
n_healthgen = cond_healthgen.sum()
n_education = cond_education.sum()
n_total = cond_total.sum()
porcentaje_total = n_total / len(df) * 100

# Mostrar resultados
print(f"Individuos con Age ≥ 60: {n_age}")
print(f"Individuos con Poverty ≤ P25 (más pobres): {n_poverty}")
print(f"Individuos con HealthGen en ['Poor']: {n_healthgen}")
print(f"Individuos con Education en ['8th Grade']: {n_education}")
print(f"\nTotal individuos que cumplen al menos una condición: {n_total} ({porcentaje_total:.2f}%)")


In [ ]:
def introducir_mar(df_original, variables, porcentaje, condicion_mask, semilla=42, verbose=True):
    np.random.seed(semilla)
    df_mar = df_original.copy()
    n_total = df_original.shape[0]

    # Subconjunto condicional (donde se permite introducir NA)
    idx_condicion = df_mar[condicion_mask].index

    for var in variables:
        # Solo filas condicionales que no tienen NA en esta variable
        idx_validos = df_mar.loc[idx_condicion][df_mar.loc[idx_condicion][var].notna()].index
        n_missing = int(n_total * porcentaje)  # total sobre todo el dataset (como en MCAR)

        idx_missing = np.random.choice(idx_validos, size=min(n_missing, len(idx_validos)), replace=False)
        df_mar.loc[idx_missing, var] = np.nan

        if verbose:
            print(f"[MAR] Introducidos NA en '{var}': {len(idx_missing)} filas ({len(idx_missing)/n_total:.2%})")

    return df_mar


In [ ]:
# Variables a las que se les simularán faltantes MAR
variables_mar = mar_num_sim_na + mar_cat_sim_na  # ya filtradas

# Máscara de condición (ya calculada previamente)
cond_mask_mar = cond_age | cond_poverty | cond_healthgen | cond_education


In [ ]:
# Aplicar faltantes MAR al 10%
df_mar_10 = introducir_mar(df, variables_mar, porcentaje=0.10, condicion_mask=cond_mask_mar, semilla=42, verbose=False)

# Aplicar faltantes MAR al 20%
df_mar_20 = introducir_mar(df, variables_mar, porcentaje=0.20, condicion_mask=cond_mask_mar, semilla=123, verbose=False)

# Aplicar faltantes MAR al 30%
df_mar_30 = introducir_mar(df, variables_mar, porcentaje=0.30, condicion_mask=cond_mask_mar, semilla=999, verbose=False)


In [ ]:
# Mostrar resumen de valores faltantes por variable
print("\nValores faltantes por variable (10% MAR):")
for var in variables_mar:
    na_count = df_mar_10[var].isnull().sum()
    na_percent = na_count / len(df_mar_10) * 100
    print(f"{var:20}: {na_count} / {len(df_mar_10)} → {na_percent:.2f}%")

# Resumen total MAR 10%
porcentaje_na_mar_10 = df_mar_10.isna().mean() * 100
na_summary_mar_10 = porcentaje_na_mar_10[porcentaje_na_mar_10 > 0].sort_values(ascending=False)

na_summary_mar_10_df = pd.DataFrame({
    "Variable": na_summary_mar_10.index,
    "Porcentaje_NA": na_summary_mar_10.values.round(2)
})
print("\nResumen de las 15 variables con más valores faltantes (%) (10% MAR):")
print(na_summary_mar_10_df.head(15))

# ------------- Repetimos para MAR 20% -------------

print("\nValores faltantes por variable (20% MAR):")
for var in variables_mar:
    na_count = df_mar_20[var].isnull().sum()
    na_percent = na_count / len(df_mar_20) * 100
    print(f"{var:20}: {na_count} / {len(df_mar_20)} → {na_percent:.2f}%")

porcentaje_na_mar_20 = df_mar_20.isna().mean() * 100
na_summary_mar_20 = porcentaje_na_mar_20[porcentaje_na_mar_20 > 0].sort_values(ascending=False)

na_summary_mar_20_df = pd.DataFrame({
    "Variable": na_summary_mar_20.index,
    "Porcentaje_NA": na_summary_mar_20.values.round(2)
})
print("\nResumen de las 15 variables con más valores faltantes (%) (20% MAR):")
print(na_summary_mar_20_df.head(15))

# ------------- MAR 30% -------------

print("\nValores faltantes por variable (30% MAR):")
for var in variables_mar:
    na_count = df_mar_30[var].isnull().sum()
    na_percent = na_count / len(df_mar_30) * 100
    print(f"{var:20}: {na_count} / {len(df_mar_30)} → {na_percent:.2f}%")

porcentaje_na_mar_30 = df_mar_30.isna().mean() * 100
na_summary_mar_30 = porcentaje_na_mar_30[porcentaje_na_mar_30 > 0].sort_values(ascending=False)

na_summary_mar_30_df = pd.DataFrame({
    "Variable": na_summary_mar_30.index,
    "Porcentaje_NA": na_summary_mar_30.values.round(2)
})
print("\nResumen de las 15 variables con más valores faltantes (%) (30% MAR):")
print(na_summary_mar_30_df.head(15))


## Simular MNAR

In [ ]:
# Copiar las listas directamente desde MCAR
mnar_num_sim_na = mcar_num_sim_na.copy()
mnar_cat_sim_na = mcar_cat_sim_na.copy()

# Unir para mostrar el total
variables_mnar = mnar_num_sim_na + mnar_cat_sim_na

# Mostrar
print(f"Variables numéricas para MNAR ({len(mnar_num_sim_na)}): {mnar_num_sim_na}")
print(f"Variables categóricas para MNAR ({len(mnar_cat_sim_na)}): {mnar_cat_sim_na}")
print(f"Total variables consideradas para MNAR: {len(variables_mnar)}")


In [ ]:
for var in mnar_cat_sim_na:
    print(f"\n📊 Variable: {var}")
    print("-" * (12 + len(var)))

    # Tabla de frecuencias absolutas y relativas
    freq = df[var].value_counts(dropna=False)
    freq_pct = df[var].value_counts(normalize=True, dropna=False) * 100

    # Combinar y mostrar
    resumen = pd.DataFrame({
        "Frecuencia": freq,
        "Porcentaje (%)": freq_pct.round(2)
    })
    print(resumen)


In [ ]:
condiciones_mnar_cat = {
    "HHIncome": ["0-4999", "5000-9999", "10000-14999"],
    "AgeDecade": ["70+"],
    "HealthGen": ["Poor", "Fair"]
}


In [ ]:
print("🧩 Categorías seleccionadas para MNAR (variables categóricas):")
for var, categorias in condiciones_mnar_cat.items():
    print(f"- {var}: {categorias}")


In [ ]:
# Estas son las variables numéricas importantes para simular MNAR
mnar_num_sim_na = [
    'BMI', 'Age', 'TotChol', 'Pulse', 'BPSysAve', 'AlcoholYear',
    'DaysPhysHlthBad', 'SleepHrsNight', 'UrineVol1', 'UrineFlow1',
    'DirectChol', 'BPDiaAve'
]

# Calculamos estadísticas resumidas para decidir si simular NA en cuartil superior o inferior
resumen_numericas = df[mnar_num_sim_na].describe(percentiles=[0.25, 0.75]).T
resumen_numericas = resumen_numericas[['min', '25%', '50%', '75%', 'max']]
resumen_numericas.columns = ['Min', 'Q1', 'Mediana', 'Q3', 'Max']

# Mostrar resultados redondeados
print("\nResumen de estadísticas para variables numéricas (MNAR):")
print(resumen_numericas.round(2))



In [ ]:
condiciones_mnar_num = {
    "BMI": "superior",
    "Age": "omitir",  # o usar si se desea
    "TotChol": "superior",
    "Pulse": "superior",
    "BPSysAve": "superior",
    "AlcoholYear": "superior",
    "DaysPhysHlthBad": "inferior",
    "SleepHrsNight": "inferior",
    "UrineVol1": "superior",
    "UrineFlow1": "superior",
    "DirectChol": "superior",
    "BPDiaAve": "superior"
}


In [ ]:
def introducir_mnar(df_original, variables_num, variables_cat,
                    condiciones_num, condiciones_cat,
                    porcentaje, semilla=42, verbose=True):

    np.random.seed(semilla)
    df_mnar = df_original.copy()
    n_total = df.shape[0]

    # --- Variables numéricas ---
    for var in variables_num:
        if condiciones_num.get(var) == "omitir":
            continue

        if condiciones_num[var] == "superior":
            mask = df_mnar[var] >= df_mnar[var].quantile(0.75)
        elif condiciones_num[var] == "inferior":
            mask = df_mnar[var] <= df_mnar[var].quantile(0.25)
        else:
            continue

        # Eliminar NA existentes
        idx_validos = df_mnar[mask & df_mnar[var].notna()].index
        n_missing = int(n_total * porcentaje)
        idx_missing = np.random.choice(idx_validos, size=min(n_missing, len(idx_validos)), replace=False)
        df_mnar.loc[idx_missing, var] = np.nan

        if verbose:
            print(f"[MNAR] {var:<15} → {len(idx_missing)} NA en región '{condiciones_num[var]}'")

    # --- Variables categóricas ---
    for var in variables_cat:
        categorias = condiciones_cat.get(var)
        if not categorias:
            continue

        mask = df_mnar[var].isin(categorias)
        idx_validos = df_mnar[mask & df_mnar[var].notna()].index
        n_missing = int(n_total * porcentaje)
        idx_missing = np.random.choice(idx_validos, size=min(n_missing, len(idx_validos)), replace=False)
        df_mnar.loc[idx_missing, var] = np.nan

        if verbose:
            print(f"[MNAR] {var:<15} → {len(idx_missing)} NA en categorías {categorias}")

    return df_mnar


In [ ]:
# Unir variables
variables_mnar_num = mnar_num_sim_na
variables_mnar_cat = mnar_cat_sim_na

df_mnar_10 = introducir_mnar(df, variables_mnar_num, variables_mnar_cat,
                             condiciones_mnar_num, condiciones_mnar_cat,
                             porcentaje=0.10, semilla=42, verbose=False)

df_mnar_20 = introducir_mnar(df, variables_mnar_num, variables_mnar_cat,
                             condiciones_mnar_num, condiciones_mnar_cat,
                             porcentaje=0.20, semilla=123, verbose=False)

df_mnar_30 = introducir_mnar(df, variables_mnar_num, variables_mnar_cat,
                             condiciones_mnar_num, condiciones_mnar_cat,
                             porcentaje=0.30, semilla=999, verbose=False)


In [ ]:
# Dataset original: número total de filas
n_filas = len(df)

# Función auxiliar para mostrar resumen por variable
def resumen_mnar(df_mnar, variables_mnar, nombre):
    print(f"\nValores faltantes por variable ({nombre}):")
    for var in variables_mnar:
        na_count = df_mnar[var].isnull().sum()
        na_percent = na_count / n_filas * 100
        print(f"{var:20}: {na_count} / {n_filas} → {na_percent:.2f}%")

    # Resumen total ordenado
    porcentaje_na = df_mnar[variables_mnar].isnull().mean() * 100
    resumen_df = porcentaje_na[porcentaje_na > 0].sort_values(ascending=False).round(2)
    resumen_df = resumen_df.reset_index()
    resumen_df.columns = ["Variable", "Porcentaje_NA"]
    print(f"\nResumen de las 15 variables con más valores faltantes (%) ({nombre}):")
    print(resumen_df.head(15))

# Unir todas las variables MNAR para revisión
variables_mnar = variables_mnar_num + variables_mnar_cat

# Aplicar resumen
resumen_mnar(df_mnar_10, variables_mnar, "10% MNAR")
resumen_mnar(df_mnar_20, variables_mnar, "20% MNAR")
resumen_mnar(df_mnar_30, variables_mnar, "30% MNAR")
